In [1]:
"""
======================================================================
CLIP retrieval captioner (CS6350 - TPA19) -- pretrained CLIP baseline
======================================================================
No training. For each test image, a pretrained CLIP model picks the caption
from a bank of human-written COCO captions whose text embedding is most
similar (cosine) to the image embedding. Scored with exactly the same metric
code and the same 5,000 val2017 test images as GIT.py / BLIP.ipynb.

Caption bank = captions of the 90% TRAIN split only (never val2017, never the
tuning-validation split), so there is no test leakage.

Usage (same folder as GIT.py, so the data paths and split files are shared):
    python clip_retrieval.py
    python clip_retrieval.py --clip_model openai/clip-vit-base-patch32

Or from a notebook:
    import clip_retrieval
    clip_retrieval.run()

Output: outputs/clip-retrieval/predictions.json and metrics.json
        (predictions.json has the same format as GIT.py's, so it works with
         compare_predictions).
======================================================================
"""

import json
import time
import argparse
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

CONFIG = {
    "clip_model": "openai/clip-vit-base-patch16",   # same ViT-B/16 family GIT-base uses
    "data_root": "./coco_data",
    "train_split_ann": "annotations/captions_train_split.json",
    "test_ann": "annotations/captions_val2017.json",
    "test_img_dir": "val2017",
    "max_test_samples": 5000,
    "output_dir": "./outputs",
    "cache_dir": "./outputs/clip_cache",
    "text_batch": 1024,
    "image_batch": 64,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}


# ----------------------------------------------------------------------
# Model loading (offline-first, like GIT.py)
# ----------------------------------------------------------------------
def load_clip(name):
    try:
        model = CLIPModel.from_pretrained(name, local_files_only=True)
        processor = CLIPProcessor.from_pretrained(name, local_files_only=True)
    except Exception:
        model = CLIPModel.from_pretrained(name)
        processor = CLIPProcessor.from_pretrained(name)
    model.to(CONFIG["device"]).eval()
    if CONFIG["device"] == "cuda":
        model.half()
    return model, processor


def _text_embed(model, input_ids, attention_mask):
    out = model.text_model(input_ids=input_ids, attention_mask=attention_mask)
    return model.text_projection(out.pooler_output)


def _image_embed(model, pixel_values):
    out = model.vision_model(pixel_values=pixel_values)
    return model.visual_projection(out.pooler_output)


# ----------------------------------------------------------------------
# Caption bank and test data
# ----------------------------------------------------------------------
def load_caption_bank():
    root = Path(CONFIG["data_root"])
    with open(root / CONFIG["train_split_ann"]) as f:
        split = json.load(f)
    caps = sorted({a["caption"].strip() for a in split["annotations"] if a["caption"].strip()})
    print(f"Caption bank: {len(caps)} unique train-split captions")
    return caps


def load_test():
    root = Path(CONFIG["data_root"])
    with open(root / CONFIG["test_ann"]) as f:
        coco = json.load(f)
    id_to_file = {img["id"]: img["file_name"] for img in coco["images"]}
    refs = {}
    for a in coco["annotations"]:
        refs.setdefault(a["image_id"], []).append(a["caption"].strip())
    image_ids = list(refs.keys())[: CONFIG["max_test_samples"]]    # same selection as GIT.evaluate
    paths = [str(root / CONFIG["test_img_dir"] / id_to_file[i]) for i in image_ids]
    return image_ids, paths, refs


# ----------------------------------------------------------------------
# Embedding
# ----------------------------------------------------------------------
@torch.no_grad()
def embed_captions(model, processor, captions):
    dev = CONFIG["device"]
    out, bs, t0 = [], CONFIG["text_batch"], time.time()
    for i in range(0, len(captions), bs):
        enc = processor.tokenizer(captions[i:i + bs], padding=True, truncation=True,
                                  max_length=77, return_tensors="pt").to(dev)
        e = _text_embed(model, enc["input_ids"], enc["attention_mask"]).float()
        out.append(torch.nn.functional.normalize(e, dim=-1).cpu())
        if (i // bs) % 50 == 0:
            print(f"  text {i}/{len(captions)}  ({time.time() - t0:.0f}s)")
    return torch.cat(out)


@torch.no_grad()
def embed_images(model, processor, paths):
    dev = CONFIG["device"]
    out, bs, t0 = [], CONFIG["image_batch"], time.time()
    for i in range(0, len(paths), bs):
        imgs = [Image.open(p).convert("RGB") for p in paths[i:i + bs]]
        pv = processor(images=imgs, return_tensors="pt")["pixel_values"].to(dev)
        if dev == "cuda":
            pv = pv.half()
        e = _image_embed(model, pv).float()
        out.append(torch.nn.functional.normalize(e, dim=-1).cpu())
        if (i // bs) % 10 == 0:
            print(f"  images {i}/{len(paths)}  ({time.time() - t0:.0f}s)")
    return torch.cat(out)


# ----------------------------------------------------------------------
# Metrics: identical code to GIT.evaluate()
# ----------------------------------------------------------------------
def compute_metrics(image_ids, paths, preds, refs_by_image):
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    from nltk.translate.meteor_score import meteor_score
    from rouge_score import rouge_scorer
    import nltk
    nltk.download("wordnet", quiet=True)
    nltk.download("omw-1.4", quiet=True)

    smoothie = SmoothingFunction().method4
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    bleu, met, rou = [], [], []
    gts, res = {}, {}
    for img_id, path in zip(image_ids, paths):
        hyp = preds[path]
        refs = refs_by_image[img_id]
        ref_tokens = [r.split() for r in refs]
        bleu.append(sentence_bleu(ref_tokens, hyp.split(), smoothing_function=smoothie))
        met.append(meteor_score(ref_tokens, hyp.split()))
        rou.append(max(scorer.score(r, hyp)["rougeL"].fmeasure for r in refs))
        gts[img_id] = refs
        res[img_id] = [hyp]

    metrics = {
        "BLEU-4": sum(bleu) / len(bleu),
        "METEOR": sum(met) / len(met),
        "ROUGE-L": sum(rou) / len(rou),
    }
    from pycocoevalcap.cider.cider import Cider
    metrics["CIDEr"], _ = Cider().compute_score(gts, res)
    return metrics


# ----------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------
def run(clip_model=None):
    clip_model = clip_model or CONFIG["clip_model"]
    tag = "clip-retrieval" if clip_model == "openai/clip-vit-base-patch16" else \
          "clip-retrieval-" + clip_model.split("/")[-1]
    out_dir = Path(CONFIG["output_dir"]) / tag
    out_dir.mkdir(parents=True, exist_ok=True)
    cache = Path(CONFIG["cache_dir"]); cache.mkdir(parents=True, exist_ok=True)
    safe = clip_model.replace("/", "_")

    print(f"Device: {CONFIG['device']} | CLIP: {clip_model}")
    model, processor = load_clip(clip_model)
    captions = load_caption_bank()
    image_ids, paths, refs_by_image = load_test()
    print(f"Retrieving captions for {len(paths)} held-out TEST images (val2017)...")

    # text embeddings are cached (the bank is large)
    tfile = cache / f"{safe}_text.npy"
    if tfile.exists():
        text_emb = torch.from_numpy(np.load(tfile))
        assert len(text_emb) == len(captions), "cache does not match caption bank; delete it"
        print("Loaded cached caption embeddings")
    else:
        print("Embedding caption bank...")
        text_emb = embed_captions(model, processor, captions)
        np.save(tfile, text_emb.numpy().astype(np.float16))
        text_emb = text_emb.half()

    print("Embedding test images...")
    img_emb = embed_images(model, processor, paths)

    # cosine similarity (embeddings are normalised) -> best caption per image
    dev = CONFIG["device"]
    text_dev = text_emb.to(dev).half()
    best = []
    for i in range(0, len(img_emb), 250):
        sims = img_emb[i:i + 250].to(dev).half() @ text_dev.T
        best.append(sims.argmax(dim=1).cpu())
    best = torch.cat(best).tolist()
    preds = {p: captions[j] for p, j in zip(paths, best)}

    for p in paths[:5]:
        print(f"{Path(p).name}: {preds[p]}")

    metrics = compute_metrics(image_ids, paths, preds, refs_by_image)
    print(f"BLEU-4 (avg, smoothed): {metrics['BLEU-4']:.4f}")
    print(f"METEOR (avg):           {metrics['METEOR']:.4f}")
    print(f"ROUGE-L (avg F1):       {metrics['ROUGE-L']:.4f}")
    print(f"CIDEr:                  {metrics['CIDEr']:.4f}")

    with open(out_dir / "predictions.json", "w") as f:
        json.dump(preds, f, indent=2)
    with open(out_dir / "metrics.json", "w") as f:
        json.dump({**metrics, "clip_model": clip_model, "caption_bank_size": len(captions)}, f, indent=2)
    print(f"Predictions + metrics saved to {out_dir}")
    return metrics

In [2]:
run()

Device: cuda | CLIP: openai/clip-vit-base-patch16


config.json: 0.00B [00:00, ?B/s]

C:\Users\iniya\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\iniya\.cache\huggingface\hub\models--openai--clip-vit-base-patch16. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin:   0%|          | 0.00/599M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

... (4 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Caption bank: 512943 unique train-split captions
Retrieving captions for 5000 held-out TEST images (val2017)...
Embedding caption bank...
  text 0/512943  (2s)
  text 51200/512943  (13s)
  text 102400/512943  (24s)
  text 153600/512943  (34s)
  text 204800/512943  (44s)
  text 256000/512943  (55s)
  text 307200/512943  (68s)
  text 358400/512943  (79s)
  text 409600/512943  (89s)
  text 460800/512943  (99s)
  text 512000/512943  (107s)
Embedding test images...
  images 0/5000  (2s)
  images 640/5000  (18s)
  images 1280/5000  (39s)
  images 1920/5000  (50s)
  images 2560/5000  (85s)
  images 3200/5000  (124s)
  images 3840/5000  (158s)
  images 4480/5000  (188s)
000000179765.jpg: A side view of a black motorcycle with a brown seat .
000000190236.jpg: Panoramic photo of desk space with three monitors.
000000331352.jpg: a toilet with a raised seats and controls on a side arm.
000000517069.jpg: A woman talks on a cell phone while setting on a bench at a bus stop on a sidewalk.
00000018241

{'BLEU-4': 0.11586998598743871,
 'METEOR': 0.3586938853674858,
 'ROUGE-L': 0.415551390867364,
 'CIDEr': np.float64(0.39342368343785594)}